# Semana 9 — Soluciones: Data Quality
**Bootcamp:** Fundamentos de Ingeniería de Datos — Databricks SQL

> **Nota:** Este notebook contiene las soluciones de los ejercicios de **Data Quality (Parte 2)** únicamente.
> Los ejercicios de Jobs/Workflows (Parte 1) y Dashboards (Parte 3) se resuelven en la UI de Databricks y no tienen solución en notebook.

---
## Parte 2: Data Quality
---

### Ejercicio 2.1 — Verificar pipeline por capas (SETUP)

In [0]:
%sql
-- Es normal que Bronze >= Silver >= Gold porque cada capa limpia y filtra datos
SELECT 'bronze' as capa, COUNT(*) as registros FROM bootcamp.bronze.properties_bronze
UNION ALL
SELECT 'silver', COUNT(*) FROM bootcamp.silver.propiedades
UNION ALL
SELECT 'gold_fact', COUNT(*) FROM bootcamp.gold.fact_propiedades
ORDER BY registros DESC;

### Ejercicio 2.2 — Integridad referencial (GUIDED)

In [0]:
%sql
-- LEFT JOIN con IS NULL detecta FKs huérfanas. 0 = integridad ok.
SELECT 'zona_id' as fk, COUNT(*) as huerfanos
FROM bootcamp.gold.fact_propiedades fp
LEFT JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
WHERE dz.zona_id IS NULL
UNION ALL
SELECT 'caracteristicas_id', COUNT(*)
FROM bootcamp.gold.fact_propiedades fp
LEFT JOIN bootcamp.gold.dim_caracteristicas dc ON fp.caracteristicas_id = dc.caracteristicas_id
WHERE dc.caracteristicas_id IS NULL
UNION ALL
SELECT 'tipo_operacion_id', COUNT(*)
FROM bootcamp.gold.fact_propiedades fp
LEFT JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
WHERE dto.tipo_operacion_id IS NULL
UNION ALL
SELECT 'orientacion_id', COUNT(*)
FROM bootcamp.gold.fact_propiedades fp
LEFT JOIN bootcamp.gold.dim_orientacion do ON fp.orientacion_id = do.orientacion_id
WHERE do.orientacion_id IS NULL;

### Ejercicio 2.3 — Validaciones de rango (INDEPENDENT)

In [0]:
%sql
-- 3 validaciones de rango con contexto de dominio inmobiliario
SELECT 'precio_negativo_o_extremo' as check_name, 
    SUM(CASE WHEN precio <= 0 OR precio > 50000000 THEN 1 ELSE 0 END) as fallos
FROM bootcamp.gold.fact_propiedades
UNION ALL
SELECT 'metros_invalidos',
    SUM(CASE WHEN metros_cuadrados_totales <= 0 OR metros_cuadrados_totales > 10000 THEN 1 ELSE 0 END)
FROM bootcamp.gold.fact_propiedades
UNION ALL
SELECT 'ambientes_irrazonables',
    SUM(CASE WHEN ambientes < 0 OR ambientes > 20 THEN 1 ELSE 0 END)
FROM bootcamp.gold.fact_propiedades;

---
## Parte 4: Reflexión — Respuestas sugeridas
---

1. **DQ como tasks del workflow:** Si las validaciones son manuales, dependen de que alguien se acuerde de ejecutarlas. Como tasks del workflow, se ejecutan automáticamente después de cada carga. Si fallan, el workflow envía una alerta y el equipo se entera inmediatamente — no cuando un analista reporta datos raros.

2. **Views semánticas para dashboards:** Si el dashboard hace JOINs directamente, cualquier cambio en el modelo dimensional rompe los dashboards. Con views semánticas, el JOIN está encapsulado: si cambia el schema, solo se actualiza la view y todos los dashboards siguen funcionando.

3. **Frecuencia de refresh:** Depende de: (a) con qué frecuencia cambian los datos subyacentes, (b) cuánto paga el negocio por datos frescos vs stale, (c) el costo de compute del refresh (cada ejecución consume recursos). Para nuestro dataset de propiedades (batch diario), un refresh diario a las 7am (después del pipeline de 6am) es suficiente.

4. **Preguntas al analista:** ¿Qué decisión necesitás tomar con estos datos? ¿Con qué frecuencia lo vas a consultar? ¿Necesitás filtrar por algo específico? ¿Quién más va a usar este dashboard? ¿Cuáles son los 3-5 números más importantes que necesitás ver?